# Process Radiomics Features

Loads per-patient PyRadiomics `.pkl` files and restructures them into per-feature-family dictionaries keyed by patient ID. Saves one `.pkl` per feature family under `Results/Analysis_Results/Radiomics/`.


In [ ]:
from tqdm import tqdm
import os
from os import listdir
import time
from random import randint
from os.path import isfile, join

from scipy.stats import skew, kurtosis
import pickle as pkl
 
import gc 
import numpy as np
from scipy import stats
import pandas as pd


import nibabel as nib

from skimage import feature

import matplotlib.pyplot as plt
from matplotlib import cm






import warnings
warnings.simplefilter("ignore")

In [ ]:
Intensity = ['diagnostics_Image-original_Mean', 
 'diagnostics_Image-original_Minimum', 
 'diagnostics_Image-original_Maximum']

print('Intensity', len(Intensity))

Size = ['diagnostics_Mask-original_VoxelNum', 
 'diagnostics_Mask-original_VolumeNum', 
 'original_shape_MeshVolume', 
 'original_shape_SurfaceArea', 
 'original_shape_SurfaceVolumeRatio', 
 'original_shape_VoxelVolume']

print('Size', len(Size))

Location = ['diagnostics_Mask-original_CenterOfMassIndex', 
 'diagnostics_Mask-original_CenterOfMass']

print('location', len(Location))

Shape = ['original_shape_Elongation', 
 'original_shape_Flatness', 
 'original_shape_LeastAxisLength', 
 'original_shape_MajorAxisLength', 
 'original_shape_Maximum2DDiameterColumn', 
 'original_shape_Maximum2DDiameterRow', 
 'original_shape_Maximum2DDiameterSlice', 
 'original_shape_MinorAxisLength', 
 'original_shape_Sphericity']

print('Shape', len(Shape))

Firstorder = ['original_firstorder_10Percentile',
'original_firstorder_90Percentile',
'original_firstorder_Energy',
'original_firstorder_Entropy',
'original_firstorder_InterquartileRange',
'original_firstorder_Kurtosis',
'original_firstorder_Maximum',
'original_firstorder_MeanAbsoluteDeviation',
'original_firstorder_Mean', 
'original_firstorder_Median',
'original_firstorder_Minimum',
'original_firstorder_Range',
'original_firstorder_RobustMeanAbsoluteDeviation',
'original_firstorder_RootMeanSquared',
'original_firstorder_Skewness', 
'original_firstorder_TotalEnergy', 
'original_firstorder_Uniformity',
'original_firstorder_Variance']

print('Firstorder', len(Firstorder))

Glcm = ['original_glcm_Autocorrelation',
'original_glcm_ClusterProminence',
'original_glcm_ClusterShade',
'original_glcm_ClusterTendency',
'original_glcm_Contrast',
'original_glcm_Correlation',
'original_glcm_DifferenceAverage', 
'original_glcm_DifferenceEntropy',
'original_glcm_DifferenceVariance',
'original_glcm_Id',
'original_glcm_Idm', 
'original_glcm_Idmn',
'original_glcm_Idn',
'original_glcm_Imc1',
'original_glcm_Imc2',
'original_glcm_InverseVariance',
'original_glcm_JointAverage',
'original_glcm_JointEnergy',
'original_glcm_JointEntropy',
'original_glcm_MCC',
'original_glcm_MaximumProbability',
'original_glcm_SumAverage',
'original_glcm_SumEntropy',
'original_glcm_SumSquares']

print('Glcm', len(Glcm))

Gldm = ['original_gldm_DependenceEntropy',
'original_gldm_DependenceNonUniformity',
'original_gldm_DependenceNonUniformityNormalized',
'original_gldm_DependenceVariance', 
'original_gldm_GrayLevelNonUniformity', 
'original_gldm_GrayLevelVariance', 
'original_gldm_HighGrayLevelEmphasis',
'original_gldm_LargeDependenceEmphasis',
'original_gldm_LargeDependenceHighGrayLevelEmphasis',
'original_gldm_LargeDependenceLowGrayLevelEmphasis',
'original_gldm_LowGrayLevelEmphasis', 
'original_gldm_SmallDependenceEmphasis',
'original_gldm_SmallDependenceHighGrayLevelEmphasis',
'original_gldm_SmallDependenceLowGrayLevelEmphasis']

print('Gldm', len(Gldm))


Glrlm = ['original_glrlm_GrayLevelNonUniformity',
'original_glrlm_GrayLevelNonUniformityNormalized',
'original_glrlm_GrayLevelVariance', 
'original_glrlm_HighGrayLevelRunEmphasis',
'original_glrlm_LongRunEmphasis',
'original_glrlm_LongRunHighGrayLevelEmphasis',
'original_glrlm_LongRunLowGrayLevelEmphasis',
'original_glrlm_LowGrayLevelRunEmphasis',
'original_glrlm_RunEntropy', 
'original_glrlm_RunLengthNonUniformity',
'original_glrlm_RunLengthNonUniformityNormalized',
'original_glrlm_RunPercentage',
'original_glrlm_RunVariance',
'original_glrlm_ShortRunEmphasis',
'original_glrlm_ShortRunHighGrayLevelEmphasis',
'original_glrlm_ShortRunLowGrayLevelEmphasis']

print('Glrlm', len(Glrlm))


Glszm = ['original_glszm_GrayLevelNonUniformity',
'original_glszm_GrayLevelNonUniformityNormalized',
'original_glszm_GrayLevelVariance',
'original_glszm_HighGrayLevelZoneEmphasis',
'original_glszm_LargeAreaEmphasis',
'original_glszm_LargeAreaHighGrayLevelEmphasis',
'original_glszm_LargeAreaLowGrayLevelEmphasis',
'original_glszm_LowGrayLevelZoneEmphasis',
'original_glszm_SizeZoneNonUniformity',
'original_glszm_SizeZoneNonUniformityNormalized',
'original_glszm_SmallAreaEmphasis',
'original_glszm_SmallAreaHighGrayLevelEmphasis',
'original_glszm_SmallAreaLowGrayLevelEmphasis',
'original_glszm_ZoneEntropy', 
'original_glszm_ZonePercentage', 
'original_glszm_ZoneVariance']

print('Glszm', len(Glszm))

Ngtdm = ['original_ngtdm_Busyness',
'original_ngtdm_Coarseness', 
'original_ngtdm_Complexity',
'original_ngtdm_Contrast',
'original_ngtdm_Strength']

print('Other', len(Ngtdm))

In [ ]:
Intensity_results = {}
for metric in Intensity:
    Intensity_results[metric] = {}

Size_results = {}
for metric in Size:
    Size_results[metric] = {}
    
Location_results = {}
for metric in Location:
    Location_results[metric] = {}
    
Shape_results = {}
for metric in Shape:
    Shape_results[metric] = {}
    
Firstorder_results = {}
for metric in Firstorder:
    Firstorder_results[metric] = {}
    
Glcm_results_1 = {}
for metric in Glcm:
    Glcm_results_1[metric] = {}

Glcm_results_5 = {}
for metric in Glcm:
    Glcm_results_5[metric] = {}
    
Glcm_results_10 = {}
for metric in Glcm:
    Glcm_results_10[metric] = {}

Gldm_results_1 = {}
for metric in Gldm:
    Gldm_results_1[metric] = {}
    
Gldm_results_5 = {}
for metric in Gldm:
    Gldm_results_5[metric] = {}
    
Gldm_results_10 = {}
for metric in Gldm:
    Gldm_results_10[metric] = {}
    
Glrlm_results = {}
for metric in Glrlm:
    Glrlm_results[metric] = {}
    
Glszm_results = {}
for metric in Glszm:
    Glszm_results[metric] = {}
    
Ngtdm_results_1 = {}
for metric in Ngtdm:
    Ngtdm_results_1[metric] = {}
    
Ngtdm_results_5 = {}
for metric in Ngtdm:
    Ngtdm_results_5[metric] = {}
    
Ngtdm_results_10 = {}
for metric in Ngtdm:
    Ngtdm_results_10[metric] = {}

In [ ]:
# Feature region: 'Tumor_Boundary' = morphological boundary of the WT mask
_type = 'Tumor_Boundary'
folder_path = '../Results/Result/Radiomics/Radiomics_Features_'+ _type + '/'
print(folder_path)
# List all files in the directory
files = os.listdir(folder_path)
modalities = ['flair', 't2', 't1', 't1ce']

# Loop through each file in the directory
for file_name in files:
    # Strip extension from filename to get patient ID key
    patient_id = file_name.split('.')[0]
    if file_name in ['.DS_Store']:
        continue
    # Construct the full file path
    file_path = os.path.join(folder_path, file_name)
    with open(file_path, 'rb') as f:
        texture_results = pkl.load(f)
        
        # Intensity
        for key in Intensity_results.keys():
            if patient_id not in Intensity_results[key]:
                Intensity_results[key][patient_id] = {'flair':0, 't2':0, 't1':0, 't1ce':0}
            for modality in modalities:
                Intensity_results[key][patient_id][modality]  = texture_results[1][modality][key]
                
        # Size
        for key in Size_results.keys():
            if patient_id not in Size_results[key]:
                Size_results[key][patient_id] = {'flair':0, 't2':0, 't1':0, 't1ce':0}
            for modality in modalities:
                Size_results[key][patient_id][modality]  = texture_results[1][modality][key]
                
        # Shape
        for key in Shape_results.keys():
            if patient_id not in Shape_results[key]:
                Shape_results[key][patient_id] = {'flair':0, 't2':0, 't1':0, 't1ce':0}
            for modality in modalities:
                Shape_results[key][patient_id][modality]  = texture_results[1][modality][key]
                
        # Firstorder
        for key in Firstorder_results.keys():
            if patient_id not in Firstorder_results[key]:
                Firstorder_results[key][patient_id] = {'flair':0, 't2':0, 't1':0, 't1ce':0}
            for modality in modalities:
                Firstorder_results[key][patient_id][modality]  = texture_results[1][modality][key]
                
        # Glcm_1
        for key in Glcm_results_1.keys():
            if patient_id not in Glcm_results_1[key]:
                Glcm_results_1[key][patient_id] = {'flair':0, 't2':0, 't1':0, 't1ce':0}
            for modality in modalities:
                Glcm_results_1[key][patient_id][modality]  = texture_results[1][modality][key]
                

                

                
        # Gldm_1
        for key in Gldm_results_1.keys():
            if patient_id not in Gldm_results_1[key]:
                Gldm_results_1[key][patient_id] = {'flair':0, 't2':0, 't1':0, 't1ce':0}
            for modality in modalities:
                Gldm_results_1[key][patient_id][modality]  = texture_results[1][modality][key]
        

        # Glrlm
        for key in Glrlm_results.keys():
            if patient_id not in Glrlm_results[key]:
                Glrlm_results[key][patient_id] = {'flair':0, 't2':0, 't1':0, 't1ce':0}
            for modality in modalities:
                Glrlm_results[key][patient_id][modality]  = texture_results[1][modality][key]     
                
        # Glszm
        for key in Glszm_results.keys():
            if patient_id not in Glszm_results[key]:
                Glszm_results[key][patient_id] = {'flair':0, 't2':0, 't1':0, 't1ce':0}
            for modality in modalities:
                Glszm_results[key][patient_id][modality]  = texture_results[1][modality][key]
                
        # Ngtdm_1
        for key in Ngtdm_results_1.keys():
            if patient_id not in Ngtdm_results_1[key]:
                Ngtdm_results_1[key][patient_id] = {'flair':0, 't2':0, 't1':0, 't1ce':0}
            for modality in modalities:
                Ngtdm_results_1[key][patient_id][modality]  = texture_results[1][modality][key]
                

In [ ]:
with open('../Results/Analysis_Results/Radiomics/' + _type + '/intensity.pkl', 'wb') as handle:
    pkl.dump(Intensity_results, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/size.pkl', 'wb') as handle:
    pkl.dump(Size_results, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/shape.pkl', 'wb') as handle:
    pkl.dump(Shape_results, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/firstorder.pkl', 'wb') as handle:
    pkl.dump(Firstorder_results, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/glcm_1.pkl', 'wb') as handle:
    pkl.dump(Glcm_results_1, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/glcm_5.pkl', 'wb') as handle:
    pkl.dump(Glcm_results_5, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/glcm_10.pkl', 'wb') as handle:
    pkl.dump(Glcm_results_10, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/gldm_1.pkl', 'wb') as handle:
    pkl.dump(Gldm_results_1, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/gldm_5.pkl', 'wb') as handle:
    pkl.dump(Gldm_results_5, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/gldm_10.pkl', 'wb') as handle:
    pkl.dump(Gldm_results_10, handle, protocol=pkl.HIGHEST_PROTOCOL)
          
with open('../Results/Analysis_Results/Radiomics/' + _type + '/glrlm.pkl', 'wb') as handle:
    pkl.dump(Glrlm_results, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/glszm.pkl', 'wb') as handle:
    pkl.dump(Glszm_results, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/ngtdm_1.pkl', 'wb') as handle:
    pkl.dump(Ngtdm_results_1, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/ngtdm_5.pkl', 'wb') as handle:
    pkl.dump(Ngtdm_results_5, handle, protocol=pkl.HIGHEST_PROTOCOL)
    
with open('../Results/Analysis_Results/Radiomics/' + _type + '/ngtdm_10.pkl', 'wb') as handle:
    pkl.dump(Ngtdm_results_10, handle, protocol=pkl.HIGHEST_PROTOCOL)
          
          
          